<a href="https://colab.research.google.com/github/mahbub19601/Enterprise_RAG_Assistant/blob/main/Enterprise_RAG_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --no-cache-dir -U langchain langchain-classic langchain-core langchain-community langchain-text-splitters langchain-openai langchain-chroma pypdf gradio

In [ ]:
import os
import logging
from typing import Optional, Tuple
import gradio as gr
from google.colab import userdata

#  LangChain Imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# API Key Setup
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except userdata.SecretNotFoundError:
    raise ValueError(" OPENAI_API_KEY is missing! Please click the 'Secrets' icon on the left, add your key, and enable notebook access.")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class RAGPipeline:
    """A professional-grade RAG pipeline using LangChain's updated architecture."""

    def __init__(self):
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.vector_store: Optional[Chroma] = None
        self.qa_chain = None

        # Define the system prompt for RAG
        system_prompt = (
            "You are a helpful AI assistant. Use the following pieces of retrieved context to answer "
            "the user's question. If the answer is not contained in the context, say that you don't know. "
            "Keep your answers concise, professional, and well-structured."
            "\n\n"
            "Context:\n{context}"
        )
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
        ])

    def process_document(self, file_path: str) -> str:
        """Loads a PDF, chunks the text, and stores it in the vector database."""
        try:
            logger.info(f"Loading document from {file_path}")
            loader = PyPDFLoader(file_path)
            docs = loader.load()

            logger.info("Splitting document into chunks.")
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200
            )
            splits = text_splitter.split_documents(docs)

            logger.info("Embedding chunks and storing in Chroma DB.")
            self.vector_store = Chroma.from_documents(documents=splits, embedding=self.embeddings)

            retriever = self.vector_store.as_retriever(search_kwargs={"k": 3})
            question_answer_chain = create_stuff_documents_chain(self.llm, self.prompt)
            self.qa_chain = create_retrieval_chain(retriever, question_answer_chain)

            return f"✅ Successfully processed {len(splits)} text chunks from the document. You can now ask questions."

        except Exception as e:
            logger.error(f"Error processing document: {str(e)}")
            return f" Error processing document: {str(e)}"

    def answer_question(self, question: str, history: list) -> Tuple[str, list]:
        """Queries the RAG chain and updates the chat history."""
        if not self.qa_chain:
            return "Please upload and process a document first.", history

        try:
            logger.info(f"Answering question: {question}")
            response = self.qa_chain.invoke({"input": question})
            answer = response["answer"]

            history.append((question, answer))
            return "", history

        except Exception as e:
            logger.error(f"Error during retrieval: {str(e)}")
            return f"Sorry, an error occurred: {str(e)}", history

# Initialize the pipeline
rag_system = RAGPipeline()

# --- Gradio UI Setup ---
def build_ui():
    """Constructs the Gradio web interface."""
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 📄 Enterprise RAG Assistant")
        gr.Markdown("Upload a PDF to dynamically inject context into an LLM using **LangChain**, **ChromaDB**, and **OpenAI**.")

        with gr.Row():
            with gr.Column(scale=1):
                file_input = gr.File(label="Upload PDF", file_types=[".pdf"])
                process_btn = gr.Button("Process Document", variant="primary")
                status_output = gr.Textbox(label="System Status", interactive=False)

            with gr.Column(scale=2):
                chatbot = gr.Chatbot(label="Chat History", height=450)
                msg_input = gr.Textbox(label="Ask a question about the document", placeholder="E.g., What are the key takeaways from page 2?")
                clear_btn = gr.ClearButton([msg_input, chatbot])

        # Event Listeners
        process_btn.click(
            fn=rag_system.process_document,
            inputs=[file_input],
            outputs=[status_output]
        )

        msg_input.submit(
            fn=rag_system.answer_question,
            inputs=[msg_input, chatbot],
            outputs=[msg_input, chatbot]
        )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    # share=True creates the public Gradio live link
    demo.launch(share=True, debug=True)

/tmp/ipykernel_8637/3409960513.py:98: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5f63008f2413ff065d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
